<span style="font-size:10pt">DIADEM course "Deep Learning - Image Classification & objects detection" -- Mai 20-21, 2026 - v1.1<br> 
CC BY-SA 4.0 Jean-Luc CHARLES (Jean-Luc.charles@mailo.com)</span>

<span style="color:Sienna; font-family:arial;font-size:1.2cm; font-weight:bold">
    Training a Convolutional Neural Network<br>to classify material micrographs
</span>    

<span style="color:Sienna; font-family:arial; font-size:15pt;">
    In this notebook we build a Convolutionnal Neural Network (CNN) a train it to classify micrographs,<br>
    then improve its performance thanks to  <b>data augmentation</b>.
</span>

# Preliminaries

## Import the required Python modules

In [ ]:
import os
# suppress tensorflow verbose warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Deep Learning modules:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Conv2D, MaxPool2D, Flatten, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.metrics import classification_report

# General modules:
import numpy as np
import matplotlib.pyplot as plt
from time import time
from pathlib import Path
from cpuinfo import get_cpu_info
import GPUtil
import sys
import random
import cv2

# local modules:
from utils.tools import (elapsed_time_since, 
                         cpu_gpu, 
                         plot_loss_accuracy, 
                         plot_proportion_bar, 
                         scan_dir, 
                         show_conf_matrix,
                         split_stratified_into_train_val_test
                         )

In [ ]:
print(f"Python    : {sys.version.split()[0]}")
print(f"tensorflow: {tf.__version__} with keras {keras.__version__}")
print(f"numpy     : {np.__version__}")
print(f"OpenCV    : {cv2.__version__}")

## Global settings:

In [ ]:
# allows to visualize the graphs directly in the cell of the N.B.
%matplotlib inline

# SEED will be used to fix the _seed_ of the random generators to have continuations
# of repeatable random numbers
SEED = 1234

tf.get_logger().setLevel('ERROR')

## Check wether GPU is available for tensorflow or not:

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Found {len(gpus)} GPU(s):")
    for gpu in gpus:
        print(f"  - {gpu.name}")
        # configure tensorflow to dynamically allocate GPU memory as needed:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPU found, only CPU.")

## Create the `model` directory

In [ ]:
print(f'{"WORKING DIRECTORY":17s}: {Path.cwd()}')
model_dir = Path("./model")
model_dir.mkdir(exist_ok=True)
print(f'{"MODEL DIRECTORY":17s}: {model_dir.absolute().as_posix()}')

# 1 - Discover the image files in the directory `../Data`

The directory `Data` is in the parent directory:

In [ ]:
list(Path('..').iterdir())

Let's define `DATA_DIR` : the relative path to the `Data` directory:

In [ ]:
DATA_DIR = Path('../Data')

Now we build the ordered list of the subdirs in `DATA_DIR`:

In [ ]:
LIST_SUB_DIR = list(DATA_DIR.iterdir())
LIST_SUB_DIR.sort()
LIST_SUB_DIR

## Check: display some images

Let's display the first JPG files in the three image sub-directories: we read the JPG files with the function `imread` of the `openCV` module, wich returns a `ndarray` of the image pixels:

In [ ]:
fig, axes = plt.subplots(1, len(LIST_SUB_DIR), layout="constrained")
fig.set_size_inches((12,3))

for subdir, axis in zip(LIST_SUB_DIR, axes):
    files = sorted(subdir.glob('*.jpg'))           # subdir.glob(*.jpg') -> all the *.jpg files in subdir
    imBGR = cv2.imread(files[0])                   # load the image file (BGR: Blue, Green, Red)
    imRGB = cv2.cvtColor(imBGR, cv2.COLOR_BGR2RGB) # convert image format from BGR to RGB
    
    # display images:
    axis.imshow(imRGB)
    axis.set_title(f'{subdir.name}\n array shape: {imRGB.shape}', fontsize=10) 
    axis.axis('off')
    axis.axis('equal')

## Check: size of the images

Let's look if the images have all the same size:

In [ ]:
shapes = []

for subdir in LIST_SUB_DIR:
    files = sorted(subdir.glob('*.jpg'))      # '*.jpg' -> all the *.jpg files in 'subdir'
    print(f'collecting image size in subdir "{subdir.name}"')
    for file in files:
        img = cv2.imread(file)
        h, w, _ = img.shape
        shapes.append((w, h))
        
# set(shapes) gives a set where every element is uniq:        
print(f'Found image sizes: {set(shapes)}')

All the images have not the same size => we will have to resize the images to a common size.

# 2 - Preprocessing of the images

__Size__:<br>
- all the images must have the same size
- the size of the images should not be too large if we want acceptable computation times...

$\leadsto$ the micrographs available are high resolution images ( 2048$\times$1536 and 2880$\times$2160 ): we will lower the image size with the fucntion `resize` of the `openCV` module.

__RGB__  or __gray tones__:<br>
If the color information of the images is not relevant for their classification, we can gain calculation time and minimize the memory footprint by converting the images into gray tones.

$\leadsto$ As the color is not the main classification information for this dataset, we will convert the RGB images to gray-tone images with the `cvtColor` function of OpenCV.

In [ ]:
w_target, h_target = 500, 400      # new image width and height

### Example:

Let's resize and convert in gray the first images of the three image sub-directories:

In [ ]:
fig, axes = plt.subplots(1, len(LIST_SUB_DIR))
fig.set_size_inches((12,3))

for subdir, axis in zip(LIST_SUB_DIR, axes):
    files = sorted(subdir.glob('*.jpg'))              # *.jpg' means: all the *.jpg files in 'subdir'
    img = cv2.imread(files[0])                        # load the first image
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)       # convert color image into grayscale image
    img = cv2.resize(img, dsize=(w_target, h_target)) # new size given as: (width, height)
    
    axis.imshow(img, cmap='gray')
    axis.set_title(f'{subdir.name}\n ndarray shape: {img.shape}', fontsize=10) 
    axis.axis('off')
    axis.axis('equal')

fig.subplots_adjust(wspace=0.15, top=0.7)

### Build the data set

Now:<br>
- we create the ndarray `images` of all the images resized and converted in gray tone<br>
- we build the corresponding `labels` ndarray.

In [ ]:
# Initiallize empty lists:
images, labels = [], []
class_label, class_rank = [], []  # the list of the class labels & ranks

for rank, subdir in enumerate(LIST_SUB_DIR):

    label = subdir.name.split('-')[1]   # remove the '1-', '2-'.. from the subdir name
    class_label.append(label)
    class_rank.append(rank)
    
    print(f'Building dataset for class_rank:{rank}, class_label: "{label}" from subdir <{subdir.name}>')
    files = list(subdir.glob("*.jpg"))     # glob('*.jpg') -> all the *.jpg files in 'subdir'
    #files.sort()
    
    for f in files:
        print(f'\r\treading file {f.name}', end='')
        img = cv2.imread(f)                               # load the first image
        img = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)        # convert color image into grayscale image
        img = cv2.resize(img, dsize=(w_target, h_target)) # new size given as: (width, height)

        # append data to the corresponding lists:
        images.append(img)
        labels.append(rank)
    print(f'\r\tdone{40*" "}')
    
# convert lists as ndarrays:    
images = np.array(images)
labels = np.array(labels)

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Verify the content of <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;class_label&thinsp;</span> and <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;class_rank&thinsp;</span>:
</span>

In [ ]:
class_label, class_rank

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Print the total number of images:
</span>

In [ ]:
len(images)

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Check the size of the first and last images:
</span>

In [ ]:
images[0].shape, images[-1].shape

Summary:

In [ ]:
print(f'{images.shape=}, {images.dtype=}')
print(f'{labels.shape=}, {labels.dtype=}')
print(f"total size of the {len(images)} images ndarray in memory: {images.size/1024/1024:.1f} MiB")

# 3 - Prepare the _train_, _valid_ and _test_ data sets

### Split the full datset into train, valid & test datasets

Thanks to the [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function of the **sklearn** module we can split the `images` and `labels` ndarrays into sub-datasets: images and labels are randomly selected but respecting the proportion of the classes in the original dataset (this is the interest of the `stratify` argument of the `train_test_split` function).

Following the state of the art in Deep Learning, we will split the full dataset into 3 sub-datasets:
- `train` for the training of the model
- `valid` for the validation during the model training
- `test` to evaluate the model score after training.

$\leadsto$ See the function `split_stratified_into_train_val_test` in the module `tools` from the `utils` directory.<br>

In [ ]:
help(split_stratified_into_train_val_test)

<span style="color:Blue; font-family:arial; font-size:11pt;">
    You can used twice the <b>ascikit-learn</b> <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;train_test_split&thinsp;</span> function or directly the <b>utils.tools</b> <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;split_stratified_into_train_val_test&thinsp;</span> function to split the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;images&thinsp;</span> dataset into:<br>
    - 70 % for the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;train dataset&thinsp;</span>span><br>
    - 10 % for the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;valid datset&thinsp;</span>span><br>
    - 20 % for the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;test dataset&thinsp;</span>span>
</span>

In [ ]:
# we use the local "split_stratified_into_train_val_test" function:
train_dataset, valid_dataset, test_dataset = split_stratified_into_train_val_test((images, labels), 
                                                                                  frac_train=0.7, 
                                                                                  frac_val=0.1,
                                                                                  frac_test=0.2, 
                                                                                  seed=SEED)

In [ ]:
# Name explicitly the image and label arrays for the 3 datasets:
train_img, train_lab = train_dataset
valid_img, valid_lab = valid_dataset
test_img, test_lab   = test_dataset

### Define global parameters:

To avoid hard-coding the number of training, valid and test images as well as the size of the images, these parameters are retrieved from the data set:
- with the shape attribute of the `train_img` and `test_img` arrays
- with the size attribute of the first training image for example


In [ ]:
NB_TRAIN_IMG = train_img.shape[0] # number of training images
NB_VALID_IMG = valid_img.shape[0] # number of validation images 
NB_TEST_IMG  = test_img.shape[0]  # number of test images
NB_PIXEL     = train_img[0].size  # number of elements (pixels) of the firts training image: 

# Display checking:
print(f"{NB_TRAIN_IMG} training images, {NB_VALID_IMG} validation images and {NB_TEST_IMG} test images")
print(f"{train_img.shape[1]}x{train_img.shape[2]}={NB_PIXEL} pixels in each image")

NB_CLASS = len(set(train_lab))     # number of classes
print(f"{NB_CLASS} classes found in the `train_lab` ndarray")

### Check the the proportion of digits in the datasets:

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Display the proportion of digits for the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;train_lab&thinsp;</span> dataset with the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;plot_proportion_bar&thinsp;</span> function:
</span>

In [ ]:
# build the dictionnary of the classes proportion in each dataset:
prop = {}
prop['train'] = [ (train_lab == i).sum() for i in range(NB_CLASS)]  # list of # of 1, # of 2... in the train dataset
plot_proportion_bar(prop, range(NB_CLASS));

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Now, display the proportion of digits for the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;valib_lab&thinsp;</span> and <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;test_lab&thinsp;</span> datasets with the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;plot_proportion_bar&thinsp;</span> function:
</span>

In [ ]:
# build the dictionnary of the classes proportion in each dataset:
prop = {}
prop['valid'] = [ (valid_lab == i).sum() for i in range(NB_CLASS)]  # list of # of 1, # of 2... in the vaild dataset
prop['test']  = [ (test_lab  == i).sum() for i in range(NB_CLASS)]  # list of # of 1, # of 2... in the test dataset
plot_proportion_bar(prop, range(NB_CLASS));

We can expect some missclassifications for the images of the classe 3, du to the small number of images in this class...

### Check: display a train image randomy selected

After the pre-processing of the raw images to build the train dataset, it is advisable to check visualy the processed images:

In [ ]:
img_rank = np.random.randint(0, len(train_img))

plt.figure(figsize=(5, 5))
plt.imshow(train_img[img_rank], cmap='gray')
plt.title(f'Image n° {img_rank}: \n class rank: {train_lab[img_rank]}, label: {class_label[train_lab[img_rank]]}',
          fontsize=10)
plt.axis('off');

### Update the dimensions of the input data for the _keras_ module:

The convolutional layers of the *kera*s module expect 4-dimensional arrays `(batch_size, height, width, depth)` by default:
- `batch_size`: number of input images in the batch,
- `height` and `width`: height and width of images (in pixels),
- `depth`: depth of the arrays (`3` for an RGB image, `1` for a grayscale image).

The shape of the images is:

In [ ]:
train_img.shape, valid_img.shape, test_img.shape

$\leadsto$  We must add a dimension (equal to `1`) after the third dimension `400`, for example with the `reshape` method of the `ndarray` arrays, without forgetting to divide the arrays by 255 to __normalize__ them:

In [ ]:
# avec  la méthode reshape des tableaux ndarray de numpy :
x_train = train_img.reshape(train_img.shape + (1,))/255.
x_valid = valid_img.reshape(valid_img.shape + (1,))/255
x_test  = test_img.reshape(test_img.shape + (1,))/255

Check:

In [ ]:
train_img.shape, x_train.shape, x_train.min(), x_train.max(),

In [ ]:
test_img.shape, x_test.shape, x_test.min(), x_test.max()

### One-hot formatting of labels

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Define 
    <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;y_train&thinsp;</span>,
    <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;y_valid&thinsp;</span> and 
    <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;y_test&thinsp;</span> as the one-hot coded version of 
    <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;train_lab&thinsp;</span>,
    <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;valib_lab&thinsp;</span> and 
    <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;test_lab&thinsp;</span>:
</span>

In [ ]:
y_train = to_categorical(train_lab)
y_valid = to_categorical(valid_lab)
y_test  = to_categorical(test_lab)

# 4 - Build the convolutional neural network

## 4.1 - The convolution operation

### Extracting features from an image with a convolution filter

The convolution of an image by a filter (also called *kernel*) consists of moving a _small 2D window_ (3x3, 5x5....) over the pixels of the image and calculating each time the _contracted tensor product_ between the elements of the filter and the pixels of the image delimited by the filter window (sum of the products term by term).<br>

The animation below illustrates the convolution of a 5x5 image by a 3x3 filter without *padding* on the edges: we obtain a new, smaller image of 3x3 pixels<br>
<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/filter_3x3.png" width="80" style="display:inline-block;">
     <img src="img/Convolution_schematic.gif" width="300" style="display:inline-block;"><br>
     [image credit: <a href="http://deeplearning.stanford.edu/tutorial">Stanford deep learning tutorial</A>]
</p>

### Padding

To maintain the size of the image, we can use a *padding* technique to create new data on the edges of the input image (by duplicating data on the edges, or adding rows and columns of 0... for example):

<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/padding.gif" width="350"><br>
     [image credit: <a href="https://medium.com/data-science/applied-deep-learning-part-4-convolutional-neural-networks-584bc134c1e2">Arden Dertat</a> ]
</p>

The goal of convolution is to extract features present in the source image: we speak of a “feature map” to designate the image produced by the convolution operation. The state of the art leads to using several convolutional filters to extract different characteristics: we can have up to several dozen convolutional filters in the same layer of the network which generate as many _feature maps_, hence the increase in data created by these convolution operations...

#### Examples of feature extraction with known convolutional filters ([Prewitt filter](https://fr.wikipedia.org/wiki/Filtre_de_Prewitt)):

As an example to understand the convolution operation, the figure below shows the *feature maps* obtained by convolving a MNIST image (a digit 7) with 4 "3x3" filters well known in image processing (Prewitt filters for contour extraction) :

<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/7_mnist_4_filtres.png" width="500"><br>
     [image credit: JLC]
</p>

We see that these filters act as *edge detection* filters: in the output images, the whitest pixels constitute what the filters detected:
- filters (a) and (c) detect lower and upper horizontal contours,
- filters (b) and (d) detect right and left vertical contours.

These very simple examples allow you to understand how the extraction of *features* from an image using convolutional filtering works.

#### From the convolutional filter to the convolutional neuron layer

The integration of convolutional filtering into the structure of the neural network gives the following organization of calculations:

- Each convolutional filter has the same coefficients for the 3 RGB image colors: for the LeNet5 network for example, each of the 6 "5x5x3" filters of the first layer has only 25 coefficients (5x5) identical for the 3 colors R, G & B.

- Each unit (convolutional neuron) of a *feature map* of layer C1 receives 75 pixels (25 red pixels $R_i$, 25 green pixels $G_i$ and 25 blue pixels $B_i$) delimited by the position of the convolutional filter in the source image.

- The neuron $k$ of a *feature map* calculates an output $y_k = F_a(\sum_{i=1}^{25}{\omega_i(R_i + G_i + B_i) - b_k})$, where $ b_k$ is the bias of the neuron $k$ and $F_a$ the activation function (very often `relu`).

- for the 6 convolutional filters of layer C1, we therefore have 6 x (25 + 1) parameters, i.e. 156 unknown parameters for this layer which will be determined by training the network.

The same pattern is used in all convolutional layers.

### Reducing the amount of information with _pooling_

*pooling* aims to reduce the amount of data to be processed. As for the convolution operation, we move a filter over the elements of the *feature map* array and at each position of the filter, we calculate a number representing all the elements selected in the filter (the maximum value, or the average....). But unlike convolution, we move the filter without overlap.<br>
In the simplified example below, the *max spool* filter transforms the 8x8 matrix into a 4x4 matrix which describes "roughly" the same information but with less data:
<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/max_pool_2x2.png" width="350"><br>
     [image credit: JLC]
</p>

## 4.2 - CNN inspired from the "LeNet5 CNN"  with tensorflow/keras    ([wiki](https://en.wikipedia.org/wiki/LeNet))

We build a convolutional NN similar to __LeNet5__ introduced in the research paper “Gradient-Based Learning Applied To Document Recognition” in 1998 by Yann LeCun, Leon Bottou, Yoshua Bengio, and Patrick Haffnerfrom.<br>
LeNet5 was originally designed for 32$\times$32 images:

<p style="text-align:center; font-style:italic; font-size:12px;">
     <img src="img/LeNet5.png"><br>
     [image credit: <a href="https://www.researchgate.net/publication/2985446_Gradient-Based_Learning_Applied_to_Document_Recognition">ReserchGate</a> ]
</p>

<span style="color:Blue; font-family:arial; font-size:11pt;">
    In the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;buil_CNN&thinsp;</span> function below, add the lines to create this CNN (you can find some help in the previous notebooks...):<br>   
- an <b>Input</b> layer to give the dimensions of the input data with the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;nb_input&thinsp;</span> parameter of the function <br>
- a <b>Conv2D</b> layer of 16 convolution filters of 5x5 with <span style="font-family:monospace;">padding='valid'</span> and a <span style="font-family:monospace;">'relu'</span> activation function<br>
- a <b>MaxPoll</b> layer of 2x2<br>
- a <b>Conv2D</b> layer of 32 convolution filters of 5x5 with <span style="font-family:monospace;">padding='valid'</span> and a <span style="font-family:monospace;">'relu'</span> activation function<br>
- a <b>MaxPoll</b> layer of 2x2<br>
- a <b>Conv2D</b> layer of 64 convolution filters of 5x5 with <span style="font-family:monospace;">padding='valid'</span> and a <span style="font-family:monospace;">'relu'</span> activation function<br>
- a <b>MaxPoll</b> layer of 2x2<br>
- a <b>Flatten</b> layer, to flatten the images into a single vector with all the pixels<br>
- a <b>Dense</b> layer, with 128 neurones and a <span style="font-family:monospace;">'relu'</span> activation function<br>
- a <b>Dropout</b> layer, taking the values of the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;dropout&thinsp;</span> and  the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;seed&thinsp;</span> parmeters of the function<br>
- a <b>Dense</b> layer (the <b>output</b> layer), of  <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;nb_class&thinsp;</span> neurones and a <span style="font-family:monospace;">'softmax'</span> activation function <br>
</span>

In [ ]:
def build_DNN(nb_input, nb_class, dropout=0., seed=None, name=''):

    if seed is not None:
        ##########################
        # Deterministic training #
        ##########################
        # 1/ set the seed of the random generators involved by tensorflow:
        tf.keras.utils.set_random_seed(seed)
        # 2/ make the tf ops determinisctic 
        # [see https://blog.tensorflow.org/2022/05/whats-new-in-tensorflow-29.html]
        tf.config.experimental.enable_op_determinism() 

    model = Sequential(name='LeNet5')
    model.add(Input(shape=nb_input))
    model.add(Conv2D(16, kernel_size=5, padding='valid', activation='relu', name='C1'))
    model.add(MaxPool2D(pool_size=2, name='S2'))
    model.add(Conv2D(32, kernel_size=5, padding='valid', activation='relu', name='C3'))
    model.add(MaxPool2D(pool_size=2, name='S4'))
    model.add(Conv2D(64, kernel_size=5, padding='valid', activation='relu', name='C5'))
    model.add(MaxPool2D(pool_size=2, name='S6'))
    model.add(Flatten())
    model.add(Dense(128, activation='relu', name='F6'))
    model.add(Dropout(dropout, seed=seed))                           
    model.add(Dense(nb_class, activation='softmax', name='Output'))
    
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    if name: model.name = name
    return model
    

In [ ]:
model = build_DNN(nb_input=x_train[0].shape, nb_class=NB_CLASS)
model.summary()

### Save the initial state of the network (structure & data)

We can save the weights of the initial untrained network (random values) and its structure with the `model.save` method. <br>
This will be useful later to re-create the network to its initial state if we want to compare successive trainings:

In [ ]:
DROPOUT = 0.2
model_init_name = f'CNN_seed-{SEED}_drop-{DROPOUT}'
model_init_file = f'{model_init_name}_init.keras'
model_init_path = model_dir / model_init_file

model = build_DNN(nb_input=x_train[0].shape, nb_class=NB_CLASS, seed=SEED, dropout=DROPOUT)
model.save(model_init_path)

In [ ]:
tree = scan_dir('model', tag='CNN', sortbydate=True)
for i, f in enumerate(tree): 
    print(f'[{i}]:', f)

# 5 - Train the network <a name="5"></a>

We train the network with `EarlyStopp` to avoid overfit:

In [ ]:
t0 = time()

###############################################
# 1 - Define the EarlyStopping callback:
###############################################
METRIC   = 'val_loss'
PATIENCE = 2

callbacks_list = [
    EarlyStopping(monitor=METRIC,      # The parameter to monitor
                  patience=PATIENCE,   # accept that 'val_accuracy' decrease 'patience' times
                  restore_best_weights=True,
                  verbose=1) ]

# load the network structure & initial weights:
print(f'Loading model from <{model_init_path}>')
model = tf.keras.models.load_model(model_init_path)

###############################################
# 2 - Deterministic tensorflow training: 
###############################################
tf.keras.utils.set_random_seed(SEED)  # sets seeds for base-python, numpy and tf
tf.config.experimental.enable_op_determinism() 

NB_EPOCH   = 25  # the max number of epochs...
BATCH_SIZE = 16

hist = model.fit(x_train, y_train,
                 validation_data=(x_valid, y_valid), 
                 epochs=NB_EPOCH,      # the total number of successive trainings
                 batch_size=BATCH_SIZE, # fragmentation of the whole dada set in batches
                 callbacks = callbacks_list)

print(elapsed_time_since(t0))

In [ ]:
plot_loss_accuracy(hist)

# 6 - Evaluate the trained network <a name="6"></a>

### Classification report

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)

# Predicting labels for test images
predict_labels = np.argmax(model.predict(x_test), axis=-1)

# Display classification report
print("Classification Report:\n", classification_report(np.argmax(y_test, axis=-1), predict_labels))

### Confusion matrix

In [ ]:
show_conf_matrix(test_lab, predict_labels, class_label, xticks_rot='vertical', figsize=(7,6));

<span style="color:maroon; font-family:arial; font-size:11pt;">
$\leadsto$ As was expected the <b>AA_2024</b> class is not as well classified as the 3 others: we  will try to make some <b>data augmentation</b> to improve the model training for this class
</span>

# 7 - Data augmentation

The idea is to create news images by transfroming ariginal images of the dataset:
- flip left-right, up-down, rotate...
- modify contrast, luminosity...

There are a lot of libraries, modules... to make data augmentation: the goal here is to understand the mechanisms involved.

## 7.1 Prepare the augmented dataset

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Complete the cell below to build the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;orig_img&thinsp;</span> array:
    
1. Define the empty list <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;orig_img&thinsp;</span>
2. Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;files&thinsp;</span>, the list of the image files in <b>3-AA_2024</b> ($\leadsto$ see section 2: usage of <b>subdir.glob("*.jpg")</b>)
3. Loop: take one image out of 2 from the  <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;files&thinsp;</span> list:
    - read, convert greyscale and resize to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;(w_target, h_target)&thinsp;</span> the image ($\leadsto$ see section 2)
    - add the image to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;orig_img&thinsp;</span>
4. Transform the list <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;orig_img&thinsp;</span> into a ndarray.

In [ ]:
subdir =  LIST_SUB_DIR[3]
print(f'Processing image subdir <{subdir}>')

orig_img = []
files = list(subdir.glob("*.jpg"))     # glob('*.jpg') -> all the *.jpg files in 'subdir'

for f in files:
    img = cv2.imread(f)                               # load the first image
    img = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)        # convert color image into grayscale image
    img = cv2.resize(img, dsize=(w_target, h_target)) # new size given as: (width, height)
    orig_img.append(img)
        
orig_img = np.array(orig_img)

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Verify the number & size of the original images taken from the <b>3-AA_2024</b> directory (should be 46 images):
</span>

In [ ]:
orig_img.shape

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Use the <a href="https://www.tensorflow.org/api_docs/python/tf/image/flip_left_right">flip_left_right</a> function of <b>tensorflow.image</b> to transform  <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;orig_img&thinsp;</span> into the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;flip_LR&thinsp;</span> ndarray:
</span>span>

In [ ]:
from tensorflow.image import flip_left_right

flip_LR = flip_left_right(orig_img)

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Check the shape of <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;flip_LR&thinsp;</span>:
</span>

In [ ]:
flip_LR.shape

<span style="color:Blue; font-family:arial; font-size:11pt;">
    With the <a href="https://numpy.org/doc/1.26/reference/generated/numpy.concatenate.html">np.concatenate</a> function of <b></b>numpy</b>, 
define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;aug_images&thinsp;</span>, the concatenation of <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;images&thinsp;</span> and <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;flip_LR.numpy()&thinsp;</span> using axis 0.<br>
Check the new shape of <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;images&thinsp;</span>.

In [ ]:
aug_images = np.concatenate((images, flip_LR.numpy()), axis=0)
aug_images.shape

<span style="color:Blue; font-family:arial; font-size:11pt;">
   - Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;new_labels&thinsp;</span>, the ndarray of (<b>'uint8'</b>) with its single dimension given by the number of images in <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;orig_img&thinsp;</span><br> 
    - Fill the array (<span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;new_labels.fill(...)&thinsp;</span>) with the class number associated with the <b>3-AA_2024</b> directory<br>
    - Display <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;new_labels&thinsp;</span>.

In [ ]:
new_labels = np.ndarray(orig_img.shape[0], dtype='uint8')
new_labels.fill(3)
new_labels

<span style="color:Blue; font-family:arial; font-size:11pt;">
    - Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;aug_labels&thinsp;</span>, the concatenation of  <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;labels&thinsp;</span> and <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;aug_labels&thinsp;</span>, using axis 0<br>
    - Check the new shape of <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;aug_labels&thinsp;</span>.
</span>

In [ ]:
aug_labels = np.concatenate((labels, new_labels), axis=0)
aug_labels.shape

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Now, you can copy/paste/modify cells from sections 3 5 & 6 to prepare the new datasets, train and evaluate the model with these new dataset:
</span>

#### Split the `aug_images` & `aug_labels` datsets into train, valid & test datasets

In [ ]:
from utils.tools import split_stratified_into_train_val_test
train_dataset, valid_dataset, test_dataset = split_stratified_into_train_val_test((aug_images, aug_labels), 
                                                                                  frac_train=0.7, 
                                                                                  frac_val=0.1,
                                                                                  frac_test=0.2, 
                                                                                  seed=SEED)
# Extract image and label arrays from the datasets:
train_img, train_lab = train_dataset
valid_img, valid_lab = valid_dataset
test_img, test_lab   = test_dataset

#### Define global parameters

In [ ]:
NB_TRAIN_IMG = train_img.shape[0] # number of training images
NB_VALID_IMG = valid_img.shape[0] # number of validation images 
NB_TEST_IMG  = test_img.shape[0]  # number of test images

NB_PIXEL     = train_img[0].size  # number of elements (pixels) of the firts training image: 

# Display checking:
print(f"{NB_TRAIN_IMG} training images, {NB_VALID_IMG} validation images and {NB_TEST_IMG} test images")
print(f"{train_img.shape[1]}x{train_img.shape[2]}={NB_PIXEL} pixels in each image")

NB_CLASS = len(set(train_lab))     # number of classes
print(f"{NB_CLASS} classes found in the `train_lab` ndarray")

#### Check the the proportion of digits in the datasets:

In [ ]:
# build the dictionnary of the classes proportion in each dataset:
prop = {}
prop['train'] = [ (train_lab == i).sum() for i in range(NB_CLASS)]  # list of # of 1, # of 2... in the train dataset
plot_proportion_bar(prop, range(NB_CLASS));

In [ ]:
# build the dictionnary of the classes proportion in each dataset:
prop = {}
prop['valid'] = [ (valid_lab == i).sum() for i in range(NB_CLASS)]  # list of # of 1, # of 2... in the vaild dataset
prop['test']  = [ (test_lab  == i).sum() for i in range(NB_CLASS)]  # list of # of 1, # of 2... in the test dataset
plot_proportion_bar(prop, range(NB_CLASS));

### Update the dimensions of the input data for the _keras_ module:

In [ ]:
# avec  la méthode reshape des tableaux ndarray de numpy :
x_train = train_img.reshape(train_img.shape + (1,))/255.
x_valid = valid_img.reshape(valid_img.shape + (1,))/255
x_test  = test_img.reshape(test_img.shape + (1,))/255

### One-hot formatting of labels

In [ ]:
y_train = to_categorical(train_lab)
y_valid = to_categorical(valid_lab)
y_test  = to_categorical(test_lab)

### Train the model with the augmented data

In [ ]:
t0 = time()

#
# 1 - Define the 'callback' EarlyStopping in the callback list:
#
METRIC   = 'val_loss'
PATIENCE = 2

callbacks_list = [
    EarlyStopping(monitor=METRIC,      # The parameter to monitor
                  patience=PATIENCE,   # accept that 'val_accuracy' decrease 'patience' times
                  restore_best_weights=True,
                  verbose=1) ]

# load the network structure & initial weights:
print(f'Loading model from <{model_init_path}>')
model = tf.keras.models.load_model(model_init_path)

#
# 2 - Deterministic tensorflow training: 
#
tf.keras.utils.set_random_seed(SEED)  # sets seeds for base-python, numpy and tf
tf.config.experimental.enable_op_determinism() 

NB_EPOCH   = 25  # the max number of epochs...
BATCH_SIZE = 16

hist = model.fit(x_train, y_train,
                 validation_data=(x_valid, y_valid), 
                 epochs=NB_EPOCH,      # the total number of successive trainings
                 batch_size=BATCH_SIZE, # fragmentation of the whole dada set in batches
                 callbacks = callbacks_list)

print(elapsed_time_since(t0))

In [ ]:
plot_loss_accuracy(hist)

### Classification report

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)

# Predicting labels for test images
predict_labels = np.argmax(model.predict(x_test), axis=-1)

# Display classification report
print("Classification Report:\n", classification_report(np.argmax(y_test, axis=-1), predict_labels))

### Confusion matrix

In [ ]:
show_conf_matrix(test_lab, predict_labels, class_label, xticks_rot='vertical', figsize=(7,6));

# Conlusion:

<span style="color:maroon; font-family:arial; font-size:12pt;">

What you have learned in this notebook:
<ul>
  <li>How to prepare a data set of images for a classification task (sections 1 & 2)</li>
  <li>How to split the image & labels datasets into the 3 train/valid/test datasets (section 3)</li>
  <li>How to build a <b>Convolutional Neural Network</b> (CNN) for the classification of images, with the <b>tensorflow</b> modules (section 4).</li>
  <li>How to train the model, display the classification and confusion matrix to evaluate  the model agiants the unseen images of the test dataset (section 5 & 6)</li>
  <li>How to improve the model performance with the <b>data augmentation approach.</b></li>
</ul>